In [1]:
!lscpu | head -25
!grep -m1 features /proc/cpuinfo
!ldconfig -p | grep -i zdnn

Architecture:                            s390x
CPU op-mode(s):                          32-bit, 64-bit
Byte Order:                              Big Endian
CPU(s):                                  3
On-line CPU(s) list:                     0-2
Vendor ID:                               IBM/S390
Model name:                              -
Machine type:                            9175
Thread(s) per core:                      1
Core(s) per socket:                      1
Socket(s) per book:                      1
Book(s) per drawer:                      1
Drawer(s):                               3
CPU dynamic MHz:                         5508
CPU static MHz:                          5508
BogoMIPS:                                28901.00
Dispatching mode:                        horizontal
Flags:                                   esan3 zarch stfle msa ldisp eimm dfp edat etf3eh highgprs te vx vxd vxe gs vxe2 vxp sort dflt vxp2 nnpa sie
Hypervisor:                              z/VM 7.3.0
Hypervis

In [3]:
import tensorflow as tf
print(tf.config.list_physical_devices())

import torch, torch_nnpa
print(torch.__version__)
print([n for n in dir(torch_nnpa) if not n.startswith("_")])

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
2.11.0+cpu
['NNPAModule', 'NNPA_DEVICE', 'Union', 'device', 'torch']


In [4]:
import time, numpy as np, tensorflow as tf
n = 4096
a = np.random.rand(n, n).astype("float32"); b = np.random.rand(n, n).astype("float32")
ta, tb = tf.constant(a), tf.constant(b)
tf.matmul(ta, tb)  # warm-up
t = time.time(); [tf.matmul(ta, tb).numpy() for _ in range(5)]; print("tf:", round(time.time()-t, 2))
t = time.time(); [a @ b for _ in range(5)]; print("numpy:", round(time.time()-t, 2))

2026-10-17 18:23:22.297746: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 67108864 exceeds 10% of free system memory.
2026-10-17 18:23:22.605427: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 67108864 exceeds 10% of free system memory.
2026-10-17 18:23:22.641567: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 67108864 exceeds 10% of free system memory.
2026-10-17 18:23:23.656669: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 67108864 exceeds 10% of free system memory.
2026-10-17 18:23:24.597618: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 67108864 exceeds 10% of free system memory.


tf: 4.73
numpy: 13.11


In [1]:
import os, tensorflow as tf
x = tf.random.normal((2048, 2048)); y = tf.matmul(x, x).numpy()
maps = open(f"/proc/{os.getpid()}/maps").read().lower()
print(sorted({l.split()[-1] for l in maps.splitlines() if "zdnn" in l}))

['/usr/local/lib/libzdnn.so.0']


2026-10-17 18:34:35.270781: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 16777216 exceeds 10% of free system memory.
2026-10-17 18:34:35.327177: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 16777216 exceeds 10% of free system memory.
2026-10-17 18:34:35.337621: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 16777216 exceeds 10% of free system memory.
2026-10-17 18:34:35.341963: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 16777216 exceeds 10% of free system memory.


In [2]:
import time, torch, torch_nnpa
print("device name:", torch_nnpa.NNPA_DEVICE)
dev = torch.device(torch_nnpa.NNPA_DEVICE)

def bench(fn, reps=5):
    fn(); t = time.time()
    for _ in range(reps): fn()
    return round(time.time() - t, 3)

n = 2048
a, b = torch.rand(n, n), torch.rand(n, n)
print("cpu :", bench(lambda: a @ b))
try:
    ad, bd = a.to(dev), b.to(dev)
    print("nnpa:", bench(lambda: (ad @ bd).cpu()))
    print("max abs diff:", (a @ b - (ad @ bd).cpu()).abs().max().item())
except Exception as e:
    print("nnpa failed:", type(e).__name__, e)

device name: None


TypeError: device() received an invalid combination of arguments - got (NoneType), but expected one of:
 * (torch.device device)
      didn't match because some of the arguments have invalid types: (!NoneType!)
 * (str type, int index = -1)
